# Multimodal Fine-Tuning: Handwritten LaTeX OCR Expert

This tutorial demonstrates how to fine-tune a state-of-the-art vision-language model, **Qwen2-VL 2B Instruct**, for document understanding and LaTeX formula transcription. We use **Unsloth** for memory-efficient QLoRA training and the **unsloth/LaTeX_OCR** dataset from the Hugging Face hub.

## 1. Install Dependencies & Setup Environment
We install the required libraries (`unsloth`, `trl`, `peft`, `bitsandbytes`, `transformers`) to run our training.

In [1]:
import os
import torch
from datasets import load_dataset
from unsloth import FastVisionModel
from trl import SFTTrainer, SFTConfig
from unsloth.trainer import UnslothVisionDataCollator

compute_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8 else torch.float16
print('Environment initialized successfully.')

[unsloth.import_fixes|WARNING]Unsloth: vLLM was built for CUDA 12 but this system has CUDA 13.0. Please reinstall vLLM with the correct CUDA version:

  uv pip install https://github.com/vllm-project/vllm/releases/download/v0.19.1/vllm-0.19.1+cu130-cp38-abi3-manylinux_2_35_aarch64.whl


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W0720 06:39:03.112000 2930308 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


W0720 06:39:03.125000 2930308 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


🦥 Unsloth Zoo will now patch everything to make training faster!


/home/lmassaron/code/sft-examples/.venv/lib/python3.12/site-packages/unsloth/import_fixes.py:1200: FutureWarning: torch._dynamo.config.inline_inbuilt_nn_modules is deprecated and does not do anything, inline_inbuilt_nn_modules is always True. It will be removed in a future version of PyTorch.
  original_setattr(self, name, value)


Environment initialized successfully.


## 2. Load Model and Processor (FastVisionModel)
We load `unsloth/Qwen2-VL-2B-Instruct` in 4-bit precision to fit within the 16GB VRAM constraint, and enable gradient checkpointing for lower VRAM footprints.

In [2]:
MODEL_ID = 'unsloth/Qwen2-VL-2B-Instruct'

model, tokenizer = FastVisionModel.from_pretrained(
    model_name=MODEL_ID,
    load_in_4bit=True,
    use_gradient_checkpointing='unsloth',
)

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    random_state=3407,
)
print('Qwen2-VL and PEFT adapters loaded successfully.')

==((====))==  Unsloth 2026.7.3: Fast Qwen2_Vl patching. Transformers: 5.14.1. vLLM: 0.19.1.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 121.689 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.13.0+cu130. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.7.1
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

Skipping model.language_model.layers.1.mlp.gate_proj: no quant_state found
Skipping model.language_model.layers.1.mlp.up_proj: no quant_state found
Skipping model.language_model.layers.1.mlp.down_proj: no quant_state found


Qwen2-VL and PEFT adapters loaded successfully.


## 3. Load and Format Dataset (LaTeX_OCR)
We load `unsloth/LaTeX_OCR` directly from Hugging Face and map the examples into a conversational format suitable for vision-language models.

In [3]:
dataset = load_dataset('unsloth/LaTeX_OCR', split='train')

# Select a subset to train and validate quickly
shuffled_dataset = dataset.shuffle(seed=42)
train_ds = shuffled_dataset.select(range(500))
eval_ds = shuffled_dataset.select(range(500, 550))

def convert_to_conversation(sample):
    conversation = [
        {
            'role': 'user',
            'content': [
                {'type': 'text', 'text': 'Write the LaTeX representation for this image.'},
                {'type': 'image'}
            ]
        },
        {
            'role': 'assistant',
            'content': [
                {'type': 'text', 'text': sample['text']}
            ]
        },
    ]
    return {
        'messages': conversation,
        'images': [sample['image']]
    }

train_mapped = train_ds.map(convert_to_conversation, remove_columns=train_ds.column_names)
eval_mapped = eval_ds.map(convert_to_conversation, remove_columns=eval_ds.column_names)
print('Dataset sample mapped successfully.')

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Dataset sample mapped successfully.


## 4. Run SFT Trainer
We configure the Hugging Face `trl` SFTTrainer with Unsloth's optimized vision data collator. We set `remove_unused_columns = False` to prevent losing the image data during training.

In [4]:
training_args = SFTConfig(
    output_dir='qwen2-vl-latex',
    dataset_text_field='text',
    max_seq_length=512,
    remove_unused_columns=False,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=5,
    max_steps=30,
    bf16=(compute_dtype == torch.bfloat16),
    logging_steps=5,
    eval_strategy='steps',
    eval_steps=10,
    save_steps=10,
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=train_mapped,
    eval_dataset=eval_mapped,
    args=training_args,
)

# Disable KV cache during training to save memory
model.config.use_cache = False

trainer.train()

# Save the fine-tuned adapter
model.save_pretrained('qwen2-vl-latex-adapter')
tokenizer.save_pretrained('qwen2-vl-latex-adapter')
print('Vision adapter successfully saved!')

Unsloth: Model does not have a default image size - using 512


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 151645, 'bos_token_id': None}.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 500 | Num Epochs = 1 | Total steps = 30
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 28,950,528 of 2,237,936,128 (1.29% trained)


Step,Training Loss,Validation Loss
10,0.501303,0.307680
20,0.280091,0.246292
30,0.137422,0.156411


Unsloth: Restored added_tokens_decoder metadata in qwen2-vl-latex/checkpoint-10/tokenizer_config.json.


Unsloth: Restored added_tokens_decoder metadata in qwen2-vl-latex/checkpoint-20/tokenizer_config.json.


Unsloth: Restored added_tokens_decoder metadata in qwen2-vl-latex/checkpoint-30/tokenizer_config.json.


Unsloth: Restored added_tokens_decoder metadata in qwen2-vl-latex-adapter/tokenizer_config.json.


Vision adapter successfully saved!


## 5. Evaluation and Inference
We switch the model to inference mode and generate a LaTeX translation for a validation image.

In [5]:
FastVisionModel.for_inference(model)

sample = eval_ds[0]
image = sample['image']
expected_latex = sample['text']

messages = [
    {
        'role': 'user',
        'content': [
            {'type': 'text', 'text': 'Write the LaTeX representation for this image.'},
            {'type': 'image', 'image': image}
        ]
    }
]

input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
inputs = tokenizer(
    image,
    input_text,
    add_special_tokens=False,
    return_tensors='pt'
).to('cuda')

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=128)

generated_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

print('--- Expected LaTeX ---')
print(expected_latex)
print('\n--- Generated LaTeX ---')
print(generated_text)

Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- Expected LaTeX ---
\Gamma _ { \sigma } + \Gamma _ { m } = \int d ^ { 2 } x [ - \frac { 1 } { 8 \pi } T r ( \partial _ { \mu } U \partial _ { \mu } U ^ { \dag } ) + \frac { 1 } { 2 } m ^ { 2 } T r ( U + U ^ { \dag } - 2 ) ] ,

--- Generated LaTeX ---
\Gamma _ { \sigma } + \Gamma _ { m } = \int d ^ { 2 } x [ - \frac { 1 } { 8 \pi } T r ( \partial _ { \mu } U \partial _ { \mu } U ^ { \dagger } ) + \frac { 1 } { 2 } m ^ { 2 } T r ( U + U ^ { \dagger } - 2 ) ] ] ,
